# Grid search de hiperparámetros — BETO + LoRA

In [ ]:
# TODO = AGREGAR SIN HASHTAGS
!git clone https://github.com/camistrika/BETO_HUMOR.git
%cd BETO_HUMOR
!pip install -e . -q

In [ ]:
!pip install -q torchao --upgrade
!pip install -q transformers peft datasets scikit-learn pyyaml

In [ ]:
# === Setup en Colab (saltar si ya estás corriendo local) ===
import os

if 'COLAB_GPU' in os.environ or os.path.exists('/content'):
    REPO_URL = 'https://github.com/TU-USUARIO/TU-REPO.git'  # <- completar
    REPO_NAME = REPO_URL.split('/')[-1].replace('.git', '')

    if not os.path.exists(f'/content/{REPO_NAME}'):
        !git clone {REPO_URL}

    %cd /content/{REPO_NAME}
    !pip install -q -r requirements.txt
    !pip install -q -e .

    # Subir el CSV manualmente si no está en el repo (datasets pesados no se versionan)
    if not os.path.exists('data/raw/haha_2019_train.csv'):
        from google.colab import files
        print('Subí haha_2019_train.csv:')
        uploaded = files.upload()
        !mkdir -p data/raw
        !mv haha_2019_train.csv data/raw/

In [ ]:
import pandas as pd
from itertools import product
from transformers import AutoTokenizer

from betohumor.config import DataConfig, BetoConfig, LoraConfig
from betohumor.utils import set_seed
from betohumor.dataset import load_and_split
from betohumor.models.lora import build_beto_lora
from betohumor.hyperparam_search import run_search
from betohumor.plots import plot_training_curves, plot_grid_search_comparison

## 1. Datos y configuraciones

In [ ]:
data_config = DataConfig()
beto_config = BetoConfig()
set_seed(data_config.seed)

df_train, df_val, df_test = load_and_split(data_config)
tokenizer = AutoTokenizer.from_pretrained(beto_config.base_model)

## 2. Definición de la grilla
`builder` recibe los params de cada combinación y arma un `LoraConfig`
propio para esa corrida — `beto_config` queda fijo (cerrado por la función).

In [ ]:
R_VALUES  = [16, 32]
LR_VALUES = [1e-4, 5e-4, 1e-3, 5e-3]

search_grid = [
    {
        'r':             r,
        'lora_alpha':    r * 2,
        'learning_rate': lr,
    }
    for r, lr in product(R_VALUES, LR_VALUES)
]

def builder(params):
    lora_config = LoraConfig(r=params['r'], lora_alpha=params['lora_alpha'])
    return build_beto_lora(beto_config, lora_config)

print(f'Total combinaciones: {len(search_grid)}')
search_grid

## 3. Correr el search

In [ ]:
results = run_search(
    builder, search_grid, beto_config, data_config,
    df_train, df_val, tokenizer, seed=data_config.seed,
    output_dir_prefix='results/checkpoints/search_lora',
)

print('\n=== RESULTADOS (ordenados por Macro F1) ===')
for r in results:
    print(f"  {r['params']} -> Macro F1: {r['macro_f1']:.4f}")

## 4. Curvas de entrenamiento de cada combinación

In [ ]:
for r in results:
    print(f"\n--- {r['run_name']} (Macro F1: {r['macro_f1']:.4f}) ---")
    plot_training_curves(r['history'])

## 5. Comparación entre configuraciones

In [ ]:
labels = [r['run_name'] for r in results]
scores = [r['macro_f1'] for r in results]
plot_grid_search_comparison(labels, scores, title='Grid search LoRA — Macro F1 en validación')

## 6. Guardar resultados en CSV
Guardado manual, a tu criterio.

In [ ]:
df_results = pd.DataFrame([
    {**r['params'], 'macro_f1': r['macro_f1'], 'run_name': r['run_name']}
    for r in results
])
df_results.to_csv('results/hyperparam_search_lora.csv', index=False)
df_results